In [ ]:
%pip install -q kagglehub "libreyolo[onnx,openvino,fast-eval]" nncf
%pip install -q --upgrade jupyter ipywidgets
# !git clone https://github.com/LuisPeregrina/gdl-atsc-anti-spillback.git
# !mv gdl-atsc-anti-spillback/* .

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import shutil
import pathlib
from pathlib import Path
from time import sleep

import torch
# torch.serialization.safe_globals([pathlib._local.PosixPath])
torch.serialization.add_safe_globals([pathlib._local.PosixPath])


import kagglehub
import yaml
from libreyolo import LibreYOLO
from libreyolo.training import TrainEndEvent, TrainEpochEvent, TrainStartEvent, TrainExceptionEvent

from tools.mtid_split_yolo import split_dataset


In [ ]:

MODEL_NAME = f"LibreYOLOs"
if "FOMO" in MODEL_NAME:
    cmd = f"curl -L https://huggingface.co/LibreYOLO/{MODEL_NAME}/resolve/main/{MODEL_NAME}.pt -o weights/{MODEL_NAME}.pt -C -"
    get_ipython().system(cmd)  # == '!curl...
    

# Configs per model
config = yaml.safe_load(Path("config.yaml").read_text())
models = config["models"]
model = next((m for m in models if m["name"] == MODEL_NAME), None)
IMAGE_SIZE = int(model["image_size"])
BATCH_SIZE = int(model["batch_size"])

DATASET_NAME = config["dataset"]["name"]
DATASET_PATH = Path.cwd() / config["dataset"]["directory"]
EPOCHS = config["epochs"]
SLEEP_MINUTES = int(config["sleep_minutes"])
LEARN_RATE = float(config["learn_rate"])

** Resuming transfer from byte position 307937
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   988  100   988    0     0   3593      0 --:--:-- --:--:-- --:--:--  3605
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [4]:
dataset_path = kagglehub.dataset_download(DATASET_NAME, output_dir=str(DATASET_PATH))
# If "Using Colab cache for faster access to the 'multiview-traffic-intersection-dataset' dataset." we need to copy from cache
#!cp -r {dataset_path} {DATASET_PATH}
yaml_path = split_dataset(dataset_path)

deduplicated 4 content-identical image(s)
total: 5772 images across train, val, test (train: 4618, val: 577, test: 577)
wrote 5772 YOLO label files beside images
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/train.list.txt (4618 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/val.list.txt (577 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/test.list.txt (577 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/mtid.yaml


Some utils so that we can interrupt and get the last weights

In [5]:
# Logger
class RunLog:
    def copy_last(self, event: TrainEpochEvent) -> None:
        fname = "last.pt"
        event_last_pt = Path(event.save_dir) / "weights" / fname
        if not event_last_pt.exists():
            print(f"Warning: {event_last_pt} does not exist, skipping copy.")
            return
        print(f"Copying {event_last_pt} to {fname}")
        shutil.copy(event_last_pt, fname)

 
    def on_train_epoch_end(self, event: TrainEpochEvent) -> None:
        if event.is_best:
            print(f"new best at epoch {event.epoch}: {event.best_metric}")

        # self.copy_last(event)
        print("Sleeping 3 minutes to cool down...")
        if SLEEP_MINUTES:
            sleep(60*SLEEP_MINUTES)

In [6]:
model = LibreYOLO(f"{MODEL_NAME}.pt")

In [9]:
resuming = False
if resuming:
    model = LibreYOLO("runs/train/fomo_exp6/weights/last.pt")

model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        workers=0,
        lr0=LEARN_RATE,
        callbacks=RunLog(),
        # resume=False
    )


2026-09-03 08:52:44 | INFO     | Using device: mps
2026-09-03 08:52:44 | INFO     | Setting up training...
2026-09-03 08:52:44 | INFO     | AutoBatch: non-CUDA device (mps) — keeping batch=16
2026-09-03 08:52:44 | INFO     | AutoBatch: per-GPU=16  world_size=1  global=16
2026-09-03 08:52:44 | INFO     | AutoBatch: resolved global batch size = 16
2026-09-03 08:52:44 | INFO     | Dataset nc=4 differs from model nc=1 — rebuilding head.
2026-09-03 08:52:44 | INFO     | FOMOLoss rebuilt with resolved dataset nc=4
2026-09-03 08:52:44 | INFO     | FOMO training dataset: 4618 images
2026-09-03 08:52:44 | INFO     | Grid size: 28×28 (imgsz=224, downsample=8)
2026-09-03 08:52:44 | INFO     | Iterations per epoch: 289 (batch_per_rank=16, world_size=1)
2026-09-03 08:52:44 | INFO     | Optimizer: adam
2026-09-03 08:52:44 | INFO     |   - pg0 (BN): 19 params
2026-09-03 08:52:44 | INFO     |   - pg1 (Conv, wd=0.0): 20 params
2026-09-03 08:52:44 | INFO     |   - pg2 (Bias): 20 params
2026-09-03 08:52:

KeyboardInterrupt: 

In [ ]:
path = model.export(format="openvino")
print(path)


In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive in Colab
try:
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    drive_root = Path.home()

# Destination folder in Drive
target_dir = drive_root / "gdl-atsc-anti-spillback" / "exports"
target_dir.mkdir(parents=True, exist_ok=True)

src = Path(path)
dst = target_dir / src.name

if src.is_dir():
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
else:
    if dst.exists():
        dst.unlink()
    shutil.copy2(src, dst)

print(f"Copied {src} -> {dst}")

In [ ]:
import inspect

print(inspect.signature(model.train))

## Interpreting FOMO points

`LibreFOMOs-point` uses FOMO, a grid-based point localizer. Each detected object is represented by the center of a low-resolution grid cell; it does not regress the car's exact center like a bounding-box detector. Therefore `points.xy` is correctly returned in original-image pixels, but its coordinates are quantized and can appear evenly spaced. The count (`len(points)`) can still be correct.

`nms_radius` only controls how close neighboring grid detections may be before suppression. To obtain more precise centers, use a higher-resolution FOMO variant (`m` or `l`) or a box detector such as `LibreYOLO9t`, then compute each box center.

In [ ]:
import cv2

from IPython.display import Image, clear_output, display


video_path = "samples/288312_tiny.mp4"

cap = cv2.VideoCapture(video_path)


if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")


try:
    while True:
        success, frame = cap.read()
        if not success:
            break

        result = model(frame)
        points = result.points

        if points is not None and len(points) > 0:
            xyn = points.xyn
            if hasattr(xyn, "detach"):
                xyn = xyn.detach().cpu().numpy()

            frame_height, frame_width = frame.shape[:2]
            for x_normalized, y_normalized in xyn:
                center = (
                    int(round(x_normalized * frame_width)),
                    int(round(y_normalized * frame_height)),
                )
                cv2.circle(
                    frame,
                    center,
                    radius=6,
                    color=(0, 0, 255),
                    thickness=-1,
                )

        ok, encoded = cv2.imencode(".jpg", frame)
        if ok:
            clear_output(wait=True)
            display(Image(data=encoded.tobytes()))

except KeyboardInterrupt:
    pass

finally:
    cap.release()
    clear_output(wait=True)

In [ ]:
# Select a representative frame from the video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {video_path}")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
frame_index = frame_count // 2
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)

success, frame = cap.read()
cap.release()

if not success:
    raise RuntimeError(f"Could not read frame {frame_index}")

# FOMO inference: points are quantized grid-cell locations.
result = model(frame)
points = result.points

# Draw the predicted grid-cell locations as red dots.
if points is not None and len(points) > 0:
    xy = points.xy
    if hasattr(xy, "detach"):
        xy = xy.detach().cpu().numpy()

    for x, y in xy:
        cv2.circle(
            frame,
            (int(round(x)), int(round(y))),
            radius=8,
            color=(0, 0, 255),
            thickness=-1,
        )

print(f"Frame: {frame_index}/{frame_count - 1}")
print(f"Detected objects: {len(points) if points is not None else 0}")
if points is not None:
    print("Predicted grid-cell locations (original-image pixels):")
    print(points.xy)
    print("Classes and confidences:")
    print(points.cls, points.conf)

ok, encoded = cv2.imencode(".jpg", frame)
if ok:
    display(Image(data=encoded.tobytes()))